# 🔬 Notebook 3: Typeahead — Deep Dive (bad → best + benchmarks)


## 🛠️ Setup

```bash
cd 06-system-designs/typeahead-autocomplete
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## Three implementations, same API

We'll build three versions of `suggest(prefix) → top-K`, from naive to production-ish,
and benchmark them on the same corpus.

| Version | Idea | Query time | Build time | Memory |
|---|---|---|---|---|
| **1. Linear scan** | Loop every term | O(N · L) | O(N) | O(N) |
| **2. Sorted + bisect** | Sort terms, binary search the prefix range | O(log N + R) | O(N log N) | O(N) |
| **3. Trie + precomputed top-K** | Tree of prefixes, top-K cached per node | **O(L)** | O(N · L) | O(N · L) nodes |

- N = number of terms, L = prefix length, R = number of terms matching the prefix.
- Trie is the fastest **at query time**, which is what we care about on the keystroke hot path.
- Build cost and memory are higher → that's why we do it in a batch job and not per request.


In [ ]:
# 🏭 Shared corpus + normalizer — reused in every version below.
import random, string, time, unicodedata, bisect
from collections import defaultdict
from heapq import nlargest

def normalize(s: str) -> str:
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    return " ".join(s.lower().strip().split())

def make_corpus(n=200_000, seed=7):
    random.seed(seed)
    # use a dict so duplicate random words get their scores summed — this way
    # all three implementations (linear, sorted+bisect, trie) produce the
    # *same* top-K and the benchmark is apples-to-apples.
    agg: dict[str, int] = {}
    for _ in range(n):
        length = random.randint(3, 12)
        w = "".join(random.choices(string.ascii_lowercase, k=length))
        agg[w] = agg.get(w, 0) + random.randint(1, 10_000)
    return list(agg.items())

CORPUS = make_corpus()
print(f"corpus size: {len(CORPUS):_} terms")


## Version 1 — linear scan (bad)

In [ ]:
def suggest_linear(corpus, prefix, k=5):
    prefix = normalize(prefix)
    hits = [(t, s) for t, s in corpus if t.startswith(prefix)]
    return nlargest(k, hits, key=lambda x: x[1])

print(suggest_linear(CORPUS, "ab", k=3))


## Version 2 — sorted list + bisect (better)

If the term list is **sorted**, all terms starting with `prefix` form a contiguous
slice. We find that slice with two binary searches, then sort the slice by score.

This is exactly how a Redis `ZRANGEBYLEX` on a sorted set finds a prefix range.


In [ ]:
class SortedIndex:
    def __init__(self, corpus):
        # sort by term ascending; store terms + parallel score list.
        pairs = sorted(corpus, key=lambda x: x[0])
        self.terms = [t for t, _ in pairs]
        self.scores = [s for _, s in pairs]

    def suggest(self, prefix: str, k=5):
        prefix = normalize(prefix)
        if not prefix:
            return []
        # all strings starting with prefix lie in [prefix, prefix + "\uffff")
        lo = bisect.bisect_left(self.terms, prefix)
        hi = bisect.bisect_left(self.terms, prefix + "\uffff")
        # then pick top-k by score in that window
        window = zip(self.terms[lo:hi], self.scores[lo:hi])
        return nlargest(k, window, key=lambda x: x[1])

IDX = SortedIndex(CORPUS)
print(IDX.suggest("ab", k=3))


## Version 3 — trie with precomputed top-K (best for query time)

A **trie** (prefix tree) shares common prefixes between terms:

```
   (root)
    /  \
   p    g
   |    |\
   y    o o
   |    | |
   th   ...
```

Each node caches its **top-K descendants**, computed once at build time.
At query time we walk `len(prefix)` edges and return the cached list.
That's the whole trick.


In [ ]:
class TrieNode:
    __slots__ = ("children", "is_end", "score", "top_k")
    def __init__(self):
        self.children: dict[str, "TrieNode"] = {}
        self.is_end = False
        self.score = 0
        self.top_k: list[tuple[str, int]] = []

class Trie:
    def __init__(self, k=5):
        self.root = TrieNode()
        self.k = k

    def add(self, term: str, freq: int = 1):
        term = normalize(term)
        node = self.root
        for ch in term:
            node = node.children.setdefault(ch, TrieNode())
        node.is_end = True
        node.score += freq

    def precompute_top_k(self):
        # post-order DFS: each node merges its own term (if any) with
        # the top_k lists of its children, keeps the best K overall.
        def dfs(node: TrieNode, prefix: str):
            merged: list[tuple[str, int]] = []
            if node.is_end:
                merged.append((prefix, node.score))
            for ch, child in node.children.items():
                dfs(child, prefix + ch)
                merged.extend(child.top_k)
            node.top_k = nlargest(self.k, merged, key=lambda x: x[1])
        dfs(self.root, "")

    def suggest(self, prefix: str):
        prefix = normalize(prefix)
        node = self.root
        for ch in prefix:
            node = node.children.get(ch)
            if node is None:
                return []
        return node.top_k

trie = Trie(k=5)
for term, freq in CORPUS:
    trie.add(term, freq)
trie.precompute_top_k()
print(trie.suggest("ab"))


## Benchmark all three

Same corpus, same prefixes, same K. The numbers will vary by machine, but the
**ratios** are what matter.


In [ ]:
def bench(fn, prefixes, repeats=5):
    t0 = time.perf_counter()
    for _ in range(repeats):
        for p in prefixes:
            fn(p)
    return (time.perf_counter() - t0) / (repeats * len(prefixes)) * 1_000_000  # µs

prefixes = ["ab", "qu", "xy", "he", "th", "ne", "go"]

t_lin  = bench(lambda p: suggest_linear(CORPUS, p, k=5), prefixes)
t_srt  = bench(lambda p: IDX.suggest(p, k=5),           prefixes)
t_trie = bench(lambda p: trie.suggest(p),               prefixes)

print(f"linear scan      : {t_lin:10_.1f} µs/query")
print(f"sorted + bisect  : {t_srt:10_.1f} µs/query")
print(f"trie + top-K     : {t_trie:10_.1f} µs/query")
print()
print(f"trie is ~{t_lin/t_trie:,.0f}× faster than naive scan")
print(f"trie is ~{t_srt/t_trie:,.0f}× faster than sorted+bisect")


> 🎯 Lesson: all three give the same answers. The trie wins at query time
> because it does **no work proportional to N** — only to the length of the prefix.
> That's exactly what you want on a per-keystroke hot path.


## Updating the trie safely

**Never** mutate a live trie under concurrent reads — locking it per update is
how you miss your latency SLO.

Instead:
1. The live trie is treated as **immutable** once built.
2. A batch job reads recent query logs and builds a **new** trie off-box.
3. The service loads the new trie and does an **atomic pointer swap**.
4. The old trie is freed once all in-flight requests finish.

This is the exact same pattern as zero-downtime DB migrations or blue/green deploys.


## Ranking beyond raw popularity

Popularity alone goes stale. Real systems blend:

- **Frequency** over the last N days (baseline).
- **Trending**: rate of change, `d(frequency)/dt`.
- **Personalization**: user history, region, language — usually an overlay
  (a small per-user trie merged with the global top-K at query time).
- **Spell correction**: if prefix has no hits, try edit-distance-1 variants.

### Freshness via decaying counter

Instead of updating trie nodes every keystroke, we decay **counts in the
aggregator** on each rebuild cycle. The decay lives in the *pipeline that
feeds the next snapshot*, not the serving trie.


In [ ]:
# 📉 Tiny exponential decay example (this lives in the aggregator, not the serving trie).
class DecayingCounter:
    def __init__(self, half_life_s=3600):
        self.half_life = half_life_s
        self.count = 0.0
        self.ts = time.time()

    def add(self, n=1):
        self._decay()
        self.count += n

    def value(self):
        self._decay()
        return self.count

    def _decay(self):
        now = time.time()
        dt = now - self.ts
        if dt > 0:
            self.count *= 0.5 ** (dt / self.half_life)
            self.ts = now

c = DecayingCounter(half_life_s=0.5)
c.add(100); time.sleep(0.25); print("after 0.25s:", round(c.value(), 2))
time.sleep(0.5);             print("after 0.75s:", round(c.value(), 2))


## Optional appendix — typo tolerance

If the user types a prefix with no matches, fall back to candidates at
**edit distance 1** from the prefix (one insertion / deletion / substitution).
Real systems use richer structures (BK-tree, Symspell), but for a prefix
autocomplete an edit-distance-1 generator gets you 80% of the way.


In [ ]:
def edits1(s, alphabet="abcdefghijklmnopqrstuvwxyz "):
    splits = [(s[:i], s[i:]) for i in range(len(s) + 1)]
    deletes    = [a + b[1:]         for a, b in splits if b]
    substitutes= [a + c + b[1:]     for a, b in splits if b for c in alphabet]
    inserts    = [a + c + b         for a, b in splits      for c in alphabet]
    return set(deletes + substitutes + inserts)

# Use a small curated trie so the demo clearly shows the fuzzy fallback in action.
fuzzy_trie = Trie(k=5)
for term, freq in [
    ("python", 900), ("python tutorial", 500),
    ("google", 800), ("google docs", 400),
    ("javascript", 700), ("java", 600),
]:
    fuzzy_trie.add(term, freq)
fuzzy_trie.precompute_top_k()

def suggest_with_fuzzy(t: Trie, prefix: str, k=5):
    hits = t.suggest(prefix)
    if hits:
        return hits
    merged: list[tuple[str, int]] = []
    for candidate in edits1(normalize(prefix)):
        merged.extend(t.suggest(candidate))
    return nlargest(k, merged, key=lambda x: x[1])

# 'gogle' is 'google' with one 'o' missing — a real edit-distance-1 typo.
print("direct 'gogle'  →", fuzzy_trie.suggest("gogle"))
print("fuzzy  'gogle'  →", suggest_with_fuzzy(fuzzy_trie, "gogle"))
print("fuzzy  'pythoon'→", suggest_with_fuzzy(fuzzy_trie, "pythoon"))
print("fuzzy  'javscr' →", suggest_with_fuzzy(fuzzy_trie, "javscr"))


## Memory & sharding (quick tour)

- One English trie with 100M distinct terms, avg 12 chars ≈ **few GB** of nodes.
  Fits on one big machine, but we need many **replicas** for QPS.
- **Shard by first 1–2 characters** for really huge corpora. Each shard is a
  full trie for its key space; the suggest service routes by prefix.
- One trie **per language** keeps them independent and small.
- For personalization, a small per-user trie is merged into the global top-K at
  query time (union + nlargest).
